In [18]:
import pandas as pd
import numpy as np
import re
import json
import os
import warnings
warnings.filterwarnings("ignore")

In [19]:
city = 'Mumbai'

df = pd.read_json(rf'../web scraping/{city}/car_dataset_{city.lower()}.json', lines=True)
df.head()

,url,car_name,Price,Registration Year,Insurance,Fuel Type,Seats,Kms Driven,RTO,Ownership,...,"<a href=""/car-faqs/mg-zs-ev/what-is-the-battery-capacity-of-mg-zs-ev-203.html"" class=""bluelink"" title=""Battery Capacity"">Battery Capacity</a>",Wireless Charging,Charger Type,Charging Time (15 A Plug Point),Charging Time (7.2 kW AC Fast Charger),Charging Time (50 kW DC Fast Charger),Petrol Mileage (ARAI),Approach Angle,Break-over Angle,Departure Angle
0,https://www.cardekho.com/used-car-details/used...,Honda Amaze,₹5.40 Lakh,2020,-,Petrol,5 Seats,"10,000 Kms",Mumbai,First Owner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://www.cardekho.com/buy-used-car-details/...,Maruti Suzuki Vitara Brezza,₹6.49 Lakh,Dec 2020,Comprehensive,Petrol,5 Seats,"24,100 Kms",Mumbai,First Owner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://www.cardekho.com/used-car-details/used...,Skoda Kushaq,₹9.45 Lakh,Nov 2021,-,Petrol,5 Seats,"37,212 Kms",Mumbai,First Owner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://www.cardekho.com/used-car-details/used...,Volkswagen Ameo,₹3.55 Lakh,Nov 2016,Third Party,Petrol,5 Seats,"35,413 Kms",Mumbai,Second Owner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://www.cardekho.com/used-car-details/used...,Maruti Suzuki Ciaz,₹6.60 Lakh,2019,-,Petrol,5 Seats,"40,000 Kms",Mumbai,First Owner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
PROCESSED_FILES_LOG = 'processed_files.txt'

def get_processed_files():
    if os.path.exists(PROCESSED_FILES_LOG):
        with open(PROCESSED_FILES_LOG, 'r') as f:
            return f.read().splitlines()
    return []

def mark_file_as_processed(json_filename):
    processed = get_processed_files()
    if json_filename not in processed:
        with open(PROCESSED_FILES_LOG, 'a') as f:
            f.write(json_filename + '\n')

In [21]:
# ─────────────────────────────────────────
json_file = f'car_dataset_{city.lower()}.json'

if json_file in get_processed_files():
    print(f"⚠️ '{json_file}' already processed — skipping!")
    df = pd.read_csv('dataset.csv')  
    print(f"Loaded existing dataset with {len(df)} rows")
    
else: 
    df['city'] = city.title()
    req_col = []
    with open('features.txt', 'r') as f:
        features = f.read().split('\n')
    for i in features:
        if i in df.columns:
            req_col.append(i)
    req_col.append('city')
    df = df[req_col]

    # Drop duplicates from new data
    before = len(df)
    df = df.drop_duplicates()
    print(f"Duplicates removed from new data: {before - len(df)} rows")

    if os.path.exists('dataset.csv') and os.path.getsize('dataset.csv') > 0:
        existing_df = pd.read_csv('dataset.csv')
        combined_df = pd.concat([existing_df, df], ignore_index=True)

        # Drop duplicates from combined data
        before = len(combined_df)
        combined_df = combined_df.drop_duplicates()
        print(f"Duplicates removed from combined data: {before - len(combined_df)} rows")

        combined_df.to_csv('dataset.csv', index=False)
        print(f"Appended {len(df)} rows. Total rows: {len(combined_df)}")
    else:
        df.to_csv('dataset.csv', index=False)
        print(f"Created new dataset.csv with {len(df)} rows")

    mark_file_as_processed(json_file)
    print(f"✅ '{json_file}' marked as processed!")

⚠️ 'car_dataset_mumbai.json' already processed — skipping!
Loaded existing dataset with 4036 rows


In [22]:
df = pd.read_csv('dataset.csv')

In [23]:
# Percentage missing in each feature
for col in df.columns:
    percent_missing = df[col].isnull().sum() / len(df)
    print(f'{col}: {percent_missing:.2%}')

car_name: 0.00%
Price: 1.31%
Registration Year: 1.44%
Kms Driven: 1.31%
Ownership: 1.44%
Fuel: 13.65%
Transmission: 1.31%
Drive Type: 13.03%
Engine: 2.11%
Power: 3.54%
Mileage: 14.69%
No. of Cylinders: 1.86%
Turbo Charger: 15.19%
Seats: 1.68%
Kerb Weight: 9.17%
Ground Clearance Unladen: 36.22%
Petrol Fuel Tank Capacity: 32.23%
Diesel Fuel Tank Capacity: 74.85%
CNG Fuel Tank Capacity: 98.29%
city: 0.00%


### Data Cleaning 

In [24]:
def fill_missing(df, col):
    df = df.copy()

    # Drop duplicates
    before = len(df)
    df = df.drop_duplicates()
    print(f"Duplicates removed: {before - len(df)} rows")

    # Handle missing values
    missing_count = df[col].isnull().sum()
    missing_pct = (missing_count / len(df)) * 100

    if missing_count == 0:
        print(f"{col} — No missing values ✅")
        return df

    print(f"{col} — Missing: {missing_count} ({missing_pct:.2f}%)")

    if missing_pct > 50:
        df = df.drop(columns=[col])
        print(f"  → Dropped column (missing > 50%)")

    elif missing_pct >= 20:
        non_null_values = df[col].dropna().values
        df.loc[df[col].isnull(), col] = np.random.choice(
            non_null_values, size=missing_count)
        print(f"  → Filled with random values")

    else:
        if df[col].dtype == 'object':
            fill_value = df[col].mode()[0]
            df[col] = df[col].fillna(fill_value)
            print(f"  → Filled with mode: {fill_value}")
        else:
            fill_value = df[col].median()
            df[col] = df[col].fillna(fill_value)
            print(f"  → Filled with median: {fill_value}")

    return df 

### Target Encoding Function

In [25]:
def weighted_target_encoding(df, feature, target, m=10):
    
    global_mean = df[target].mean()
    
    stats = df.groupby(feature)[target].agg(['mean', 'count'])
    
    stats['weighted_mean'] = (
        stats['mean'] * stats['count'] + global_mean * m
    ) / (stats['count'] + m)
    
    mapping = stats['weighted_mean'].to_dict()
    
    encoded_feature = df[feature].map(mapping)
    
    return encoded_feature, mapping

In [26]:
df['city'].value_counts()

city
Mumbai       1547
Bangalore    1421
Ahmedabad    1068
Name: count, dtype: int64

0. Price

In [27]:
def clean_price(x):
    x = str(x).replace("₹", "").strip()

    if "Lakh" in x:
        return float(x.replace("Lakh", "").strip()) * 100000

    elif "Crore" in x:
        return float(x.replace("Crore", "").strip()) * 10000000

    elif "Thousand" in x:
        return float(x.replace("Thousand", "").strip()) * 1000

    else:
        return None


df["Price"] = df["Price"].apply(clean_price)

In [28]:
df = fill_missing(df, 'Price') 

Duplicates removed: 0 rows
Price — Missing: 53 (1.31%)
  → Filled with median: 595000.0


0. City

In [29]:
df['city'], city_mapping = weighted_target_encoding(
    df,
    feature='city',
    target='Price',
    m=20
)

In [30]:
df['city'].value_counts()

city
1.274597e+06    1547
1.042306e+06    1421
5.989348e+05    1068
Name: count, dtype: int64

1. 'car_name'

In [31]:
df = fill_missing(df, 'car_name')

Duplicates removed: 0 rows
car_name — No missing values ✅


In [32]:
def clean_car_name(df, city):
    # remove extra spaces
    df['car_name'] = df['car_name'].str.strip()

    # remove rows that contain scraping phrases
    pattern = rf"used cars for sale|in {city}|₹"
    df = df[~df['car_name'].str.contains(pattern, case=False, na=False)]

    # keep only names that start with letters
    df = df[df['car_name'].str.match(r'^[A-Za-z]', na=False)]

    return df

In [33]:
df = clean_car_name(df, city)

df[['brand', 'model']] = df['car_name'].str.split(' ', n=1, expand=True)

In [34]:
df['model'].nunique()

267

In [35]:
df['model'], model_mapping = weighted_target_encoding(
    df,
    feature="model",
    target="Price",
    m=20
)

df['brand'], brand_mapping = weighted_target_encoding(
    df,
    feature="brand",
    target="Price",
    m=20
)

In [36]:
df['model'].nunique()

259

In [37]:
invalid_brands = [
    'Era', 'HTX', 'LXI', 'S', 'Sportz', 'VX', 'VXI', 'XZ', 'i'
]

df = df[~df['brand'].isin(invalid_brands)]

In [38]:
df = df.drop(columns=['car_name'])

2. Registration Year

In [39]:
def clean_registration_year(x):
    if pd.isna(x):
        return None
    
    x = str(x)
    
    # extract 4-digit year
    match = re.search(r'\b(19|20)\d{2}\b', x)
    
    if match:
        return int(match.group())
    
    return None

In [40]:
df["Registration Year"] = df["Registration Year"].apply(clean_registration_year)

In [41]:
df = fill_missing(df, 'Registration Year')

Duplicates removed: 14 rows
Registration Year — Missing: 7 (0.18%)
  → Filled with median: 2018.0


3. 'Kms Driven'

In [42]:
df["Kms Driven"] = (
    df["Kms Driven"]
    .str.replace("Kms", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors='coerce')  # ✅ skips NaN, converts to float
)

In [43]:
df = fill_missing(df, 'Kms Driven')

Duplicates removed: 0 rows
Kms Driven — Missing: 2 (0.05%)
  → Filled with median: 56854.0


4. 'Ownership'

In [44]:
ownership_map = {
    "First Owner": 1,
    "Second Owner": 2,
    "Third Owner": 3,
    "Fourth Owner": 4,
    "Fifth Owner": 5
}

df["Ownership"] = df["Ownership"].map(ownership_map)
df = fill_missing(df, 'Ownership')

Duplicates removed: 0 rows
Ownership — Missing: 7 (0.18%)
  → Filled with median: 1.0


5. 'Fuel'

In [45]:
df = fill_missing(df, 'Fuel')

Duplicates removed: 0 rows
Fuel — Missing: 497 (12.56%)
  → Filled with mode: Petrol


In [46]:
# get sorted fuel categories
fuel_categories = sorted(df['Fuel'].dropna().unique())

print("All Fuel Categories:", fuel_categories)

# first category will be dropped
dropped_category = fuel_categories[0]
print("Dropped Category:", dropped_category)

# apply one hot encoding
fuel_dummies = pd.get_dummies(df['Fuel'], prefix="Fuel")

# drop the first category column
fuel_dummies.drop(f"Fuel_{dropped_category}", axis=1, inplace=True)

# merge with dataframe
df = pd.concat([df, fuel_dummies], axis=1)

# remove original column
df.drop(columns=['Fuel'], inplace=True)

All Fuel Categories: ['CNG', 'Diesel', 'Petrol']
Dropped Category: CNG


6. 'Transmission'

In [47]:
df = fill_missing(df, 'Transmission')

transmission_map = {
    "Automatic": 1,
    "Manual": 0
}

df["Transmission"] = df["Transmission"].map(transmission_map)


Duplicates removed: 0 rows
Transmission — Missing: 2 (0.05%)
  → Filled with mode: Manual


7. 'Drive Type'

In [48]:
def clean_drive_type(x):
    if pd.isna(x):
        return None
    
    x = str(x).strip().lower()

    if "fwd" in x or "front" in x:
        return "FWD"
    
    elif "rwd" in x:
        return "RWD"
    
    elif "awd" in x or "4wd" in x or "4x4" in x:
        return "AWD"
    
    elif "2wd" in x or "2 wd" in x or "two wheel" in x or "4x2" in x:
        return "FWD"
    
    else:
        return None


df["Drive Type"] = df["Drive Type"].apply(clean_drive_type)

In [49]:
df = fill_missing(df, 'Drive Type')

Duplicates removed: 0 rows
Drive Type — Missing: 480 (12.13%)
  → Filled with mode: FWD


In [50]:
Drive_Type__categories = sorted(df['Drive Type'].dropna().unique())

print("All Fuel Categories:", Drive_Type__categories)

# first category will be dropped
dropped_category = Drive_Type__categories[0]
print("Dropped Category:", dropped_category)

# apply one hot encoding
Drive_Type_dummies = pd.get_dummies(df['Drive Type'], prefix="Drive_Type")

# drop the first category column
Drive_Type_dummies.drop(f"Drive_Type_{dropped_category}", axis=1, inplace=True)

# merge with dataframe
df = pd.concat([df, Drive_Type_dummies], axis=1)

# remove original column
df.drop(columns=['Drive Type'], inplace=True)

All Fuel Categories: ['AWD', 'FWD', 'RWD']
Dropped Category: AWD


8. 'Engine 

In [51]:
df["Engine"] = (
    df["Engine"]
    .str.replace("cc", "", regex=False)
    .str.strip()
    .astype(float)
)
df = fill_missing(df, 'Engine')

Duplicates removed: 0 rows
Engine — Missing: 34 (0.86%)
  → Filled with median: 1248.0


9. Power

In [52]:
df["Power"] = (
    df["Power"]
    .str.replace("bhp", "", regex=False)
    .str.strip()
    .astype(float)
)
df = fill_missing(df, 'Power')

Duplicates removed: 0 rows
Power — Missing: 92 (2.33%)
  → Filled with median: 100.6


10. Mileage

In [53]:
df["Mileage"] = (
    df["Mileage"]
    .str.replace(r"[^\d.]", "", regex=True)
    .astype(float)
)

df = fill_missing(df, 'Mileage')

Duplicates removed: 0 rows
Mileage — Missing: 529 (13.37%)
  → Filled with median: 18.65


11. No. of Cylinders

In [54]:
df = fill_missing(df, 'No. of Cylinders')

Duplicates removed: 0 rows
No. of Cylinders — Missing: 24 (0.61%)
  → Filled with median: 4.0


12. 'Turbo Charger'

In [55]:
df = fill_missing(df, 'Turbo Charger')
tubo_map = {
    "Yes": 1,
    "No": 0
}
df['Turbo Charger'] = df['Turbo Charger'].map(tubo_map)

Duplicates removed: 0 rows
Turbo Charger — Missing: 557 (14.08%)
  → Filled with mode: No


13. Seats

In [56]:
df = fill_missing(df, 'Seats')

df["Seats"] = (
    df["Seats"]
    .str.replace("Seats", "", regex=False)
    .str.strip()
    .astype(float)
)

Duplicates removed: 0 rows
Seats — Missing: 17 (0.43%)
  → Filled with mode: 5 Seats


14. Kerb Weight

In [57]:
df["Kerb Weight"] = (
    df["Kerb Weight"]
    .str.replace(r"[^\d]", "", regex=True)
)

# convert to numeric
df["Kerb Weight"] = pd.to_numeric(df["Kerb Weight"], errors="coerce")

# fill missing values
df = fill_missing(df, 'Kerb Weight')

Duplicates removed: 0 rows
Kerb Weight — Missing: 320 (8.09%)
  → Filled with median: 1200.0


15. Ground Clearance Unladen

In [58]:
df["Ground Clearance Unladen"] = (
    df["Ground Clearance Unladen"]
    .str.replace("mm", "", regex=False)
    .str.strip()
    .astype(float)
)

df = fill_missing(df, 'Ground Clearance Unladen')

Duplicates removed: 0 rows
Ground Clearance Unladen — Missing: 1397 (35.31%)
  → Filled with random values


16. ---

In [59]:
cols = [
    "Petrol Fuel Tank Capacity",
    "Diesel Fuel Tank Capacity",
    "CNG Fuel Tank Capacity"
]

for col in cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(r"[^\d.]", "", regex=True)  # keep numbers and decimal
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [60]:
petrol_median = df.loc[df["Fuel_Petrol"] == 1, "Petrol Fuel Tank Capacity"].median()

diesel_median = df.loc[df["Fuel_Diesel"] == 1, "Diesel Fuel Tank Capacity"].median()

cng_median = df.loc[
    (df["Fuel_Diesel"] == 0) & (df["Fuel_Petrol"] == 0),
    "CNG Fuel Tank Capacity"
].median()

In [61]:
df.loc[df["Fuel_Petrol"] == 1, "Petrol Fuel Tank Capacity"] = \
df.loc[df["Fuel_Petrol"] == 1, "Petrol Fuel Tank Capacity"].fillna(petrol_median)

In [62]:
df.loc[df["Fuel_Diesel"] == 1, "Diesel Fuel Tank Capacity"] = \
df.loc[df["Fuel_Diesel"] == 1, "Diesel Fuel Tank Capacity"].fillna(diesel_median)

In [63]:
df.loc[
    (df["Fuel_Diesel"] == 0) & (df["Fuel_Petrol"] == 0),
    "CNG Fuel Tank Capacity"
] = df.loc[
    (df["Fuel_Diesel"] == 0) & (df["Fuel_Petrol"] == 0),
    "CNG Fuel Tank Capacity"
].fillna(cng_median)

In [64]:
df[cols] = df[cols].fillna(0)

---

In [65]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV, train_test_split  
from sklearn.metrics import r2_score 

In [66]:
bool_features = ['Fuel_Diesel', 'Fuel_Petrol', 'Drive_Type_FWD', 'Drive_Type_RWD'] 

df[bool_features] = df[bool_features].astype(int)

X = df.drop(columns=['Price'])  # replace with your target column name
y = df['Price']

In [67]:
print(X.dtypes[X.dtypes == 'object'])

Series([], dtype: object)


In [68]:
X = df.drop(columns=['Price'])  
y = df['Price']
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
) 

In [ ]:
param_grid = {
    'n_estimators': [200, 300, 500],
    'max_depth': [2, 3, 4],          
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.8, 1.0],   
    'reg_alpha': [0, 0.1, 0.5],             
    'reg_lambda': [1, 1.5, 2]             
}
grid_search = GridSearchCV(
    XGBRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
) 
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

# Takes around 10 minutes

In [70]:
print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

y_pred = best_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
print("R2 Score:", r2)

Best Parameters: {'colsample_bytree': 0.6, 'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 500, 'reg_alpha': 0, 'reg_lambda': 1, 'subsample': 0.6}
Best CV Score: 0.8785312276057601
R2 Score: 0.908432648320002


In [71]:
import os
import joblib

os.makedirs("model", exist_ok=True)

joblib.dump(best_model, "model/price_model.pkl")
joblib.dump(brand_mapping, "model/brand_mapping.pkl")
joblib.dump(model_mapping, "model/model_mapping.pkl")
joblib.dump(city_mapping, 'model/city_mapping.pkl')

print("Saved at:", os.path.abspath("model"))

Saved at: d:\IMP  ML  PROJECTS\CAR PRICE PREDICTION\price\model


In [72]:
X_train.columns

Index(['Registration Year', 'Kms Driven', 'Ownership', 'Transmission',
       'Engine', 'Power', 'Mileage', 'No. of Cylinders', 'Turbo Charger',
       'Seats', 'Kerb Weight', 'Ground Clearance Unladen',
       'Petrol Fuel Tank Capacity', 'Diesel Fuel Tank Capacity',
       'CNG Fuel Tank Capacity', 'city', 'brand', 'model', 'Fuel_Diesel',
       'Fuel_Petrol', 'Drive_Type_FWD', 'Drive_Type_RWD'],
      dtype='object')